# SpectraShift Week 6: aggregate curves and approve Week 7
Use CPU with Internet off. Attach source v5, `spectrashift-week5-complete`, `spectrashift-week6-contracts`, and all three Week 6 seed datasets. Direct datasets or downloaded result ZIPs are accepted.


In [ ]:
from pathlib import Path
import hashlib, json, os, shutil, sys, yaml

INPUT = Path('/kaggle/input')
projects = [p.parent for p in INPUT.rglob('pyproject.toml') if (p.parent / 'src/spectrashift/train/week6.py').is_file()]
if not projects:
    bundles = sorted(INPUT.rglob('spectrashift-kaggle-source.zip'))
    assert len(bundles) == 1, f'Expected one Week 6 source bundle, found {bundles}'
    source_work = Path('/tmp/spectrashift-week6-source')
    if source_work.exists():
        shutil.rmtree(source_work)
    shutil.unpack_archive(str(bundles[0]), str(source_work))
    projects = [source_work]
assert projects, 'No Week 6 source tree found'
projects.sort(key=lambda path: (0 if 'spectrashift-source' in str(path) else 1, len(str(path))))
PROJECT = projects[0]
sys.path.insert(0, str(PROJECT / 'src'))
os.chdir(PROJECT)

def unique_file(name):
    candidates = sorted(INPUT.rglob(name))
    by_hash = {}
    for path in candidates:
        digest = hashlib.sha256(path.read_bytes()).hexdigest()
        by_hash.setdefault(digest, path)
    assert len(by_hash) == 1, f'Expected one unique {name}; found {candidates}'
    return next(iter(by_hash.values()))

WORK = Path('/kaggle/working/spectrashift-week6-complete')
WORK.mkdir(parents=True, exist_ok=True)
archive_root = Path('/tmp/spectrashift-week6-aggregate-inputs')
if archive_root.exists():
    shutil.rmtree(archive_root)
archive_root.mkdir(parents=True)
for index, archive in enumerate(sorted(INPUT.rglob('*.zip'))):
    if archive.name != 'spectrashift-kaggle-source.zip':
        shutil.unpack_archive(str(archive), str(archive_root / str(index)))
roots = [INPUT, archive_root]

def candidates(name):
    return sorted({path for root in roots for path in root.rglob(name)})

week5_candidates = candidates('week5_run_summary.json')
week6_contract_candidates = candidates('week6_contracts_summary.json')
seed_candidates = candidates('week6_seed*_summary.json')
assert len(week5_candidates) == 1, week5_candidates
assert len(week6_contract_candidates) == 1, week6_contract_candidates
seed_groups = {}
for path in seed_candidates:
    payload = json.loads(path.read_text())
    if payload.get('week6_seed_complete') and payload.get('evaluation_labels_loaded') is False:
        seed_groups.setdefault(int(payload['seed']), []).append(path)
assert set(seed_groups) == {17, 29, 43}, f'Expected complete seeds 17, 29, 43; found {sorted(seed_groups)}'
SEED_SUMMARIES = []
for seed in (17, 29, 43):
    variants = seed_groups[seed]
    hashes = {hashlib.sha256(path.read_bytes()).hexdigest() for path in variants}
    assert len(hashes) == 1, f'Conflicting summaries for seed {seed}: {variants}'
    SEED_SUMMARIES.append(variants[0])
WEEK5_SUMMARY = week5_candidates[0]
WEEK6_CONTRACTS = week6_contract_candidates[0]
print({'seed_summaries': [str(path) for path in SEED_SUMMARIES]})


In [ ]:
from spectrashift.train.week6 import aggregate_week6

summary = aggregate_week6(WEEK5_SUMMARY, WEEK6_CONTRACTS, SEED_SUMMARIES, WORK)
print(json.dumps(summary, indent=2))
assert summary['week6_complete'] and summary['week7_approved']
assert summary['new_run_count'] == 48
assert summary['controlled_run_count'] == 90 and summary['rgb_control_run_count'] == 3
assert summary['evaluation_labels_loaded'] is False
